# Shape-guided optimisation of vortioxetine

Optimise vortioxetine using **only** the three fragment-level operations:

| Operation | Function | What it does |
|---|---|---|
| Ring replacement | `replace_ring` | Swaps entire ring systems for corpus alternatives |
| Scaffold decoration | `decorate_scaffold` | Adds substituents to free positions |
| Substituent replacement | `replace_substituent` | Swaps non-ring side chains |

**Objective**: find molecules that are *spatially similar* to vortioxetine but *graphically diverse* — i.e. scaffold-hopping candidates.

**Scoring function**:
```
score = shape * (1 - α * TS)
```
where `shape` is the `rdShapeAlign` score vs the reference conformer, `TS` is the ECFP4 Tanimoto similarity to the reference, and `α` (default 0.5) controls how strongly graph similarity is penalised. The multiplicative form ensures a molecule must earn its shape score — low Tanimoto alone cannot compensate for poor shape.

No GA loop is used — each operation is called directly each iteration, candidates are scored and the best molecules are kept as parents for the next round (beam search).


In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw, rdDistGeom, rdShapeAlign, AddHs, SDMolSupplier, DataStructs
from rdkit.Chem.Draw import MolsToGridImage
from IPython.display import display
import matplotlib.pyplot as plt
import os

RDLogger.DisableLog('rdApp.*')

from lacan import lacan
from lacan.replace import replace_ring, replace_substituent, replace_linker, decorate_scaffold, load_corpus

# ── Load profile and corpus ───────────────────────────────────────────────────
profile  = lacan.load_profile('chembl')
corpus   = load_corpus(min_count=200)

print('Profile and corpus loaded.')


## 1. Load reference molecule

Read the vortioxetine SDF, extract the 3D conformer, and compute the baseline score (self-alignment — should be near 1.0).


In [ ]:
NB_DIR = os.path.dirname(os.path.abspath('__file__'))
SDF_PATH = os.path.join(NB_DIR, 'vortioxetine.sdf')

ref_mol = next(m for m in SDMolSupplier(SDF_PATH) if m is not None)

# ref_3d: the SDF conformer with explicit Hs added in-place.
# This is what AlignMol uses as the fixed target — keeping the original
# geometry means the same molecule scores 1.0 against itself.
ref_3d = AddHs(ref_mol, addCoords=True)

# ECFP4 fingerprint of the reference — used to penalise graph similarity
ref_fp = AllChem.GetMorganFingerprintAsBitVect(ref_mol, radius=2, nBits=2048)

ref_smi  = Chem.MolToSmiles(ref_mol)
ref_score, _ = lacan.score_mol(ref_mol, profile)

print(f'Vortioxetine SMILES : {ref_smi}')
print(f'Heavy atoms         : {ref_mol.GetNumHeavyAtoms()}')
print(f'LACAN score         : {ref_score:.3f}')
display(Draw.MolToImage(ref_mol, size=(350, 280)))


## 2. Scoring function

The combined score balances 3D shape similarity against 2D graph diversity:

```
score = shape * (1 - α * TS)
```

- `shape` — best `rdShapeAlign` score across 20 conformers (colour + shape average, 0–1)
- `TS` — ECFP4 Tanimoto similarity to the reference (0–1)
- `α` — diversity penalty weight (default 0.5; raise toward 1.0 for stronger scaffold-hopping pressure)

The multiplicative form means a molecule must earn its score through shape — a low-Tanimoto molecule with poor shape still scores poorly. At `α=0.5`, a perfect shape match at TS=0.0 scores 1.0, while at TS=0.6 it scores 0.70.


In [ ]:
# ── Tunable parameter ────────────────────────────────────────────────────────
ALPHA = 0.5   # diversity penalty weight; raise toward 1.0 for more scaffold-hopping


def ecfp4_tanimoto(mol, ref_fp=ref_fp):
    """ECFP4 Tanimoto similarity between mol and the reference."""
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    return DataStructs.TanimotoSimilarity(ref_fp, fp)


def get_alignment_scores(mols, ref=ref_3d):
    """Shape-align each mol to ref using multiple conformers.

    Embeds numConfs=20 conformers per molecule, aligns each with
    rdShapeAlign.AlignMol, and returns the best average of
    (colour_score + shape_score) / 2 across conformers.

    Parameters
    ----------
    mols : list of RDKit Mol (2D)
    ref  : RDKit Mol with a 3D conformer — use ref_3d (SDF + AddHs)

    Returns
    -------
    list of float  (raw shape scores, before diversity penalty)
    """
    scores = []
    for m in mols:
        try:
            m = AddHs(m)
            rdDistGeom.EmbedMultipleConfs(m, numConfs=20, randomSeed=0xf00d, numThreads=-1)
            conf_scores = [
                sum(rdShapeAlign.AlignMol(ref, m, probeConfId=cid)) / 2
                for cid in range(m.GetNumConformers())
            ]
            scores.append(max(conf_scores) if conf_scores else 0.0)
        except Exception:
            scores.append(0.0)
    return scores


def score_batch(mols, alpha=ALPHA):
    """Combined shape + diversity score for a list of molecules.

    score = shape * (1 - alpha * TS)
    """
    shape_scores = get_alignment_scores(mols)
    return [
        shape * (1 - alpha * ecfp4_tanimoto(m))
        for shape, m in zip(shape_scores, mols)
    ]


def shape_score(mol):
    """Combined score for a single molecule."""
    return score_batch([mol])[0]


# Sanity checks
ref_shape = get_alignment_scores([ref_mol])[0]
ref_ts    = ecfp4_tanimoto(ref_mol)   # 1.0 — same molecule
ref_combined = shape_score(ref_mol)
print(f'Reference shape score    : {ref_shape:.4f}')
print(f'Reference Tanimoto (self): {ref_ts:.4f}')
print(f'Reference combined score : {ref_combined:.4f}  (= {ref_shape:.3f} * (1 - {ALPHA}*{ref_ts:.1f}))')
print(f'\nWith alpha={ALPHA}, a perfect shape match scores:')
for ts in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    print(f'  TS={ts:.1f} → combined={ref_shape * (1 - ALPHA * ts):.4f}')


## 3. Beam search optimisation

Each iteration:
1. For every parent, apply only the **same operation that created it** — a molecule produced by ring replacement can only generate further ring replacements, never decoration or substituent swaps.
2. The seed (no ancestry) spawns all three operation lineages independently.
3. Score all novel candidates; keep the top `beam_width` per lineage as parents for the next round.
4. Track winners (combined score ≥ `win_threshold`).

This guarantees every result in the output is a **pure** product of exactly one operation type.


In [ ]:
OPS = {
    'ring':   lambda mol: replace_ring(
        mol, profile, score_threshold=0.0, fragcorpus=corpus, n_replacements=25),
    'deco':   lambda mol: decorate_scaffold(
        mol, profile, score_threshold=0.0, fragcorpus=corpus,
        n_replacements=25, mode='Hydrogen'),
    'sub':    lambda mol: replace_substituent(
        mol, profile, score_threshold=0.0, fragcorpus=corpus, n_replacements=25),
    'linker': lambda mol: replace_linker(
        mol, profile, score_threshold=0.0, fragcorpus=corpus, n_replacements=25),
}


def optimize_shape(
        seed_mol,
        profile,
        corpus,
        generations      = 10,
        beam_width       = 6,
        n_replacements   = 25,
        win_threshold    = 0.50,
        score_threshold  = 0.0,
        ops              = None,
        protect_smarts   = None,
        verbose          = True,
):
    """Beam-search optimisation with single-operation lineage locking.

    The seed spawns three independent lineages (ring / deco / sub). Each
    descendant can only be mutated by the same operation that created it,
    so every result is a pure product of exactly one operation type.

    Parameters
    ----------
    seed_mol        : RDKit Mol
    profile         : LACAN profile dict
    corpus          : fragment corpus list
    generations     : int
    beam_width      : int — top molecules kept per lineage each round
    n_replacements  : int — attempts per parent per operation
    win_threshold   : float — combined score qualifying as a hit
    score_threshold : float — minimum LACAN score for products
    verbose         : bool

    Returns
    -------
    beams   : dict op -> list of (smiles, score)
    winners : list of (smiles, score, op) sorted best-first
    history : list of dicts with per-generation stats
    """
    if ops is None:
        ops = OPS

    # protect_smarts is passed to each operation call — no mol pre-marking needed.
    # The only thing we need here is to ensure the seed SMILES roundtrip is clean.
    seed_smi = Chem.MolToSmiles(seed_mol)
    seen     = {seed_smi}
    history  = []
    winners  = []

    # One beam per operation lineage: (smi, combined_score)
    beams = {op: [(seed_smi, shape_score(seed_mol))] for op in OPS}

    if verbose:
        seed_raw_shape = get_alignment_scores([seed_mol])[0]
        seed_ts = ecfp4_tanimoto(seed_mol)
        seed_sc = beams['ring'][0][1]
        print(f'Seed: {seed_smi}')
        print(f'Seed shape={seed_raw_shape:.4f}  TS={seed_ts:.4f}  combined={seed_sc:.4f}')
        print()

    for gen in range(generations):
        gen_all_shapes, gen_all_ts = [], []
        gen_winners = 0
        gen_candidates = 0

        for op_name, op_fn in ops.items():
            beam = beams[op_name]
            candidates = []

            for parent_smi, _ in beam:
                parent_mol = Chem.MolFromSmiles(parent_smi)
                if parent_mol is None:
                    continue
                try:
                    prods = op_fn(parent_mol)
                except Exception:
                    prods = []
                for m in prods:
                    try:
                        smi = Chem.MolToSmiles(m)
                    except Exception:
                        continue
                    if smi not in seen:
                        seen.add(smi)
                        candidates.append((smi, m))

            if not candidates:
                continue

            smis, mols = zip(*candidates)
            scores = score_batch(list(mols))
            gen_candidates += len(candidates)

            for smi, sc, m in zip(smis, scores, mols):
                gen_all_shapes.append(get_alignment_scores([m])[0])
                gen_all_ts.append(ecfp4_tanimoto(m))
                if sc >= win_threshold:
                    winners.append((smi, sc, op_name))
                    gen_winners += 1

            new_entries = list(beam) + list(zip(smis, scores))
            new_entries.sort(key=lambda x: x[1], reverse=True)
            beams[op_name] = new_entries[:beam_width]

        if gen_candidates == 0:
            if verbose:
                print(f'  Gen {gen+1}: no novel candidates — stopping early.')
            break

        best_per_op = {op: beams[op][0][1] for op in OPS}
        history.append({
            'generation':   gen + 1,
            'n_candidates': gen_candidates,
            'n_winners':    gen_winners,
            'best_per_op':  best_per_op,
            'best_score':   max(best_per_op.values()),
            'mean_shape':   sum(gen_all_shapes) / len(gen_all_shapes) if gen_all_shapes else 0,
            'mean_ts':      sum(gen_all_ts) / len(gen_all_ts) if gen_all_ts else 0,
        })

        if verbose:
            op_summary = '  '.join(f'{op}={best_per_op[op]:.4f}' for op in OPS)
            print(
                f'  Gen {gen+1}/{generations} | '
                f'candidates={gen_candidates:4d} | winners={gen_winners:3d} | '
                f'mean_shape={history[-1]["mean_shape"]:.3f} | '
                f'mean_TS={history[-1]["mean_ts"]:.3f}\n'
                f'    best: {op_summary}'
            )

    # Deduplicate winners, preserve op label
    seen_w, deduped = set(), []
    for smi, sc, op in sorted(winners, key=lambda x: x[1], reverse=True):
        if smi not in seen_w:
            seen_w.add(smi)
            deduped.append((smi, sc, op))

    return beams, deduped, history


## 4. Run the optimisation

Optimise from the vortioxetine seed. The win threshold applies to the **combined** score `shape * (1 - α * TS)`. Since the seed's own combined score is penalised by TS=1.0, the threshold of 0.50 targets analogues with good shape but meaningfully lower graph similarity.


In [ ]:
beams, winners, history = optimize_shape(
    ref_mol,
    profile,
    corpus,
    generations     = 10,
    beam_width      = 6,
    n_replacements  = 25,
    win_threshold   = 0.50,
    score_threshold = 0.0,
    verbose         = True,
)

print(f'\nTotal winners (combined ≥ 0.50): {len(winners)}')
for op in OPS:
    op_wins = [w for w in winners if w[2] == op]
    print(f'  {op:4s}: {len(op_wins)} winners  |  '
          f'best beam = {beams[op][0][1]:.4f}  ({beams[op][0][0]})')


## 5. Results — top beam members


In [ ]:
# Show top beam members per operation
for op in OPS:
    beam = beams[op]
    print(f'\n── {op} lineage ──')
    for i, (smi, sc) in enumerate(beam[:3]):
        m = Chem.MolFromSmiles(smi)
        raw_shape = get_alignment_scores([m])[0]
        ts        = ecfp4_tanimoto(m)
        print(f'  {i+1}. combined={sc:.4f}  shape={raw_shape:.4f}  TS={ts:.3f}  {smi}')

    top_mols   = [Chem.MolFromSmiles(smi) for smi, _ in beam[:6]]
    top_labels = [
        f'comb={sc:.3f} TS={ecfp4_tanimoto(Chem.MolFromSmiles(smi)):.2f}'
        for smi, sc in beam[:6]
    ]
    print(f'Top beam — {op}:')
    display(MolsToGridImage(
        top_mols, molsPerRow=3, subImgSize=(300, 240),
        legends=top_labels
    ))


## 6. Winners (combined ≥ 0.50) — grouped by operation


In [ ]:
if winners:
    print(f'{len(winners)} total winner(s)\n')
    for op in OPS:
        op_wins = [(smi, sc) for smi, sc, o in winners if o == op]
        if not op_wins:
            print(f'  {op}: no winners')
            continue
        print(f'── {op} ({len(op_wins)} winners) ──')
        for i, (smi, sc) in enumerate(op_wins[:9]):
            m = Chem.MolFromSmiles(smi)
            raw_shape = get_alignment_scores([m])[0]
            ts        = ecfp4_tanimoto(m)
            print(f'  {i+1}. combined={sc:.4f}  shape={raw_shape:.4f}  TS={ts:.3f}  {smi}')
        win_mols   = [Chem.MolFromSmiles(smi) for smi, _ in op_wins[:9]]
        win_labels = [
            f'comb={sc:.3f} TS={ecfp4_tanimoto(Chem.MolFromSmiles(smi)):.2f}'
            for smi, sc in op_wins[:9]
        ]
        display(MolsToGridImage(
            win_mols, molsPerRow=3, subImgSize=(300, 240),
            legends=win_labels,
        ))
else:
    print('No winners above threshold. Lower win_threshold or increase generations/n_replacements.')
    print('Best per lineage:')
    for op, beam in beams.items():
        smi, sc = beam[0]
        ts = ecfp4_tanimoto(Chem.MolFromSmiles(smi))
        print(f'  {op}: combined={sc:.4f}  TS={ts:.3f}  {smi}')


## 7. Progress plot


In [ ]:
if history:
    gens        = [h['generation']   for h in history]
    mean_shapes = [h['mean_shape']   for h in history]
    mean_ts     = [h['mean_ts']      for h in history]
    ncands      = [h['n_candidates'] for h in history]
    nwins       = [h['n_winners']    for h in history]

    op_colours = {'ring': 'steelblue', 'deco': 'seagreen', 'sub': 'darkorange', 'linker': 'orchid'}

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Panel 1: best combined per lineage
    for op, colour in op_colours.items():
        bests_op = [h['best_per_op'][op] for h in history]
        axes[0].plot(gens, bests_op, 'o-', color=colour, lw=2, label=op)
    axes[0].axhline(0.50, color='tomato', lw=1.2, ls='--', label='win threshold')
    axes[0].set_xlabel('Generation'); axes[0].set_ylabel('Best combined score')
    axes[0].set_title(f'Best combined per lineage  (α={ALPHA})')
    axes[0].legend(); axes[0].set_ylim(0, 1.05)

    # Panel 2: mean shape vs mean TS
    axes[1].plot(gens, mean_shapes, 's-', color='seagreen',  lw=2, label='Mean shape')
    axes[1].plot(gens, mean_ts,     '^-', color='darkorange', lw=2, label='Mean TS')
    axes[1].set_xlabel('Generation'); axes[1].set_ylabel('Score')
    axes[1].set_title('Mean shape vs mean Tanimoto per generation')
    axes[1].legend(); axes[1].set_ylim(0, 1.05)

    # Panel 3: candidate counts
    axes[2].bar(gens, ncands, color='slategray', alpha=0.55, label='Candidates')
    ax2r = axes[2].twinx()
    ax2r.plot(gens, nwins, 's--', color='tomato', lw=1.5, label='Winners')
    axes[2].set_xlabel('Generation')
    axes[2].set_ylabel('Candidates'); ax2r.set_ylabel('Winners')
    axes[2].set_title('Candidates and winners per generation')
    lines1, labs1 = axes[2].get_legend_handles_labels()
    lines2, labs2 = ax2r.get_legend_handles_labels()
    axes[2].legend(lines1 + lines2, labs1 + labs2, loc='upper right')

    plt.tight_layout()
    plt.show()


## 8. Per-operation breakdown (single generation diagnostic)

Run each operation once on the seed independently to inspect how many candidates each produces.


In [ ]:
diag_mol = ref_mol
N = 30

ring_out = replace_ring(diag_mol, profile, 0.0, fragcorpus=corpus, n_replacements=N)
sub_out  = replace_substituent(diag_mol, profile, 0.0, fragcorpus=corpus, n_replacements=N)
deco_out = decorate_scaffold(diag_mol, profile, 0.0, fragcorpus=corpus, n_replacements=N, mode='Hydrogen')

print(f'replace_ring        → {len(ring_out):3d} products')
print(f'replace_substituent → {len(sub_out):3d} products')
print(f'decorate_scaffold   → {len(deco_out):3d} products')

# Score and decompose best per operation
for label, prods in [('ring', ring_out), ('sub', sub_out), ('deco', deco_out)]:
    if not prods:
        print(f'  {label}: no products')
        continue
    combined = score_batch(prods)
    shapes   = get_alignment_scores(prods)
    ts_vals  = [ecfp4_tanimoto(m) for m in prods]
    best_idx = max(range(len(combined)), key=lambda i: combined[i])
    print(f'  Best {label}: combined={combined[best_idx]:.4f}  '
          f'shape={shapes[best_idx]:.4f}  TS={ts_vals[best_idx]:.3f}  '
          f'{Chem.MolToSmiles(prods[best_idx])}')


## 9. 3D overlay — best compounds vs reference

Align the top compounds to the reference conformer and visualise them together using `py3Dmol`.
The reference is shown in **white sticks**; each analogue gets a distinct colour.


In [ ]:
import py3Dmol
from rdkit.Chem import MolToMolBlock

# ── Palette — one colour per operation ───────────────────────────────────────
OP_COLOURS = {'ring': '#4C9BE8', 'deco': '#4CE87A', 'sub': '#E8824C', 'linker': '#E84C9B'}

def best_aligned_molblock(mol_2d, ref=ref_3d, n_confs=20, seed=0xf00d):
    """Embed mol_2d, align all confs to ref, return molblock of the best conf."""
    m = AddHs(mol_2d)
    rdDistGeom.EmbedMultipleConfs(m, numConfs=n_confs, randomSeed=seed, numThreads=-1)
    if m.GetNumConformers() == 0:
        return None, 0.0
    best_sc, best_cid = -1, 0
    for cid in range(m.GetNumConformers()):
        sc = sum(rdShapeAlign.AlignMol(ref, m, probeConfId=cid)) / 2
        if sc > best_sc:
            best_sc, best_cid = sc, cid
    rw = Chem.RWMol(m)
    for c in list(rw.GetConformers()):
        if c.GetId() != best_cid:
            rw.RemoveConformer(c.GetId())
    return MolToMolBlock(rw), best_sc


# ── Choose best winner from each operation ────────────────────────────────────
# Take the top-scoring winner per op; fall back to best beam member if no winners
overlay_entries = []   # list of (smi, op)
for op in OPS:
    op_wins = [(smi, sc) for smi, sc, o in winners if o == op]
    if op_wins:
        overlay_entries.append((op_wins[0][0], op))
    else:
        overlay_entries.append((beams[op][0][0], op))

print(f'Overlaying {len(overlay_entries)} analogue(s) + reference')
for smi, op in overlay_entries:
    ts = ecfp4_tanimoto(Chem.MolFromSmiles(smi))
    print(f'  [{op}] TS={ts:.3f}  colour={OP_COLOURS[op]}  {smi}')

# ── Build py3Dmol viewer ──────────────────────────────────────────────────────
view = py3Dmol.view(width=700, height=500)

# Reference — white sticks
view.addModel(MolToMolBlock(ref_3d), 'mol')
view.setStyle({'model': 0}, {'stick': {'color': 'white', 'radius': 0.12}})

# Analogues — colour-coded by operation
for i, (smi, op) in enumerate(overlay_entries):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    molblock, sc = best_aligned_molblock(mol)
    if molblock is None:
        continue
    colour = OP_COLOURS[op]
    view.addModel(molblock, 'mol')
    view.setStyle({'model': i + 1}, {'stick': {'color': colour, 'radius': 0.12}})

view.zoomTo()
view.show()


## 10. Protected optimisation — lock the piperazine pharmacophore

Re-runs the optimisation with the **piperazine ring (including both N atoms) protected**.
This locks the NH-piperazine pharmacophore in place — mutations can only touch the
tolyl ring, the thioether, and the aniline phenyl.

Protection is set via SMARTS (`N1CCNCC1`) so it is robust to index shifts across
different input molecules.

The `OPS` dict is rebuilt with `score_threshold=0.0` so the protection gates
in `replace.py` can do their job without the LACAN score pre-filter interfering.
Winners are shown grouped by operation alongside their Tanimoto similarity and
a note of which atoms remain protected, and the 3D overlay shows the best hit
from each lineage over the vortioxetine reference conformer.


In [ ]:
# ── Define the pharmacophore to lock ─────────────────────────────────────────
# Protect the piperazine ring (N1CCNCC1) — both N atoms + four CH2 groups.
# This is passed as protect_smarts= to all operations, which re-derive the
# protected atom set from each molecule on every call — no mol properties needed.
from lacan.protect import get_protected_atoms

PROTECT_SMARTS = 'N1CCNCC1'

prot_ids = sorted(get_protected_atoms(ref_mol, PROTECT_SMARTS))
print(f'Protected atom indices ({len(prot_ids)} atoms): {prot_ids}')
print('Symbols:', [ref_mol.GetAtomWithIdx(i).GetSymbol() for i in prot_ids])

# Draw seed highlighting which atoms will be locked
from lacan.protect import protect_bonds_for_idx  # bond protection unchanged
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import HTML

def draw_protected_smarts(mol, smarts, size=(450, 300), label=''):
    """Draw mol with atoms matching smarts highlighted red."""
    pa = sorted(get_protected_atoms(mol, smarts))
    atom_cols = {i: (0.85, 0.15, 0.15) for i in pa}
    atom_rads = {i: 0.4 for i in pa}
    d = rdMolDraw2D.MolDraw2DSVG(*size)
    d.drawOptions().addAtomIndices = True
    rdMolDraw2D.PrepareMolForDrawing(mol)
    d.DrawMolecule(mol,
        highlightAtoms=pa, highlightAtomColors=atom_cols, highlightAtomRadii=atom_rads)
    d.FinishDrawing()
    svg = d.GetDrawingText()
    if label:
        svg = svg.replace('</svg>',
            f'<text x="10" y="{size[1]-8}" font-size="13" fill="#444">{label}</text></svg>')
    return HTML(svg)

display(draw_protected_smarts(ref_mol, PROTECT_SMARTS,
                              label='Piperazine protected (red) — passed as SMARTS'))

# ── Build OPS_PROT with protect_smarts baked in ───────────────────────────────
OPS_PROT = {
    'ring':   lambda mol: replace_ring(
        mol, profile, score_threshold=0.0, fragcorpus=corpus,
        n_replacements=25, protect_smarts=PROTECT_SMARTS),
    'deco':   lambda mol: decorate_scaffold(
        mol, profile, score_threshold=0.0, fragcorpus=corpus,
        n_replacements=25, mode='Hydrogen', protect_smarts=PROTECT_SMARTS),
    'sub':    lambda mol: replace_substituent(
        mol, profile, score_threshold=0.0, fragcorpus=corpus,
        n_replacements=25, protect_smarts=PROTECT_SMARTS),
    'linker': lambda mol: replace_linker(
        mol, profile, score_threshold=0.0, fragcorpus=corpus,
        n_replacements=25, protect_smarts=PROTECT_SMARTS),
}

# ── Run protected optimisation ────────────────────────────────────────────────
# No pre-marking needed: protect_smarts is re-applied to each parent mol
# inside optimize_shape before every operation call.
print('\nRunning protected optimisation...')
beams_prot, winners_prot, history_prot = optimize_shape(
    ref_mol,
    profile,
    corpus,
    generations     = 10,
    beam_width      = 6,
    n_replacements  = 25,
    win_threshold   = 0.50,
    score_threshold = 0.0,
    ops             = OPS_PROT,
    protect_smarts  = PROTECT_SMARTS,
    verbose         = True,
)

print(f'\nTotal winners: {len(winners_prot)}')
for op in OPS_PROT:
    op_wins = [(smi, sc) for smi, sc, o in winners_prot if o == op]
    beam_best = beams_prot[op][0][1] if beams_prot[op] else float('nan')
    print(f'  {op:6s}: {len(op_wins)} winners  beam_best={beam_best:.4f}')


### 10a. Results — protected beam members and winners


In [ ]:
# Show top beam per lineage with protection audit
for op in OPS_PROT:
    beam = beams_prot[op]
    print(f'\n── {op} lineage ──')
    for smi, sc in beam[:3]:
        m = Chem.MolFromSmiles(smi)
        pa  = get_protected_atoms(m, PROTECT_SMARTS)
        ts  = ecfp4_tanimoto(m)
        raw_shape = get_alignment_scores([m])[0]
        print(f'  combined={sc:.4f}  shape={raw_shape:.4f}  TS={ts:.3f}  '
              f'protected={len(pa)} atoms  {smi}')

    top_mols   = [Chem.MolFromSmiles(smi) for smi, _ in beam[:6]]
    top_labels = [
        f'comb={sc:.3f} TS={ecfp4_tanimoto(Chem.MolFromSmiles(smi)):.2f}'
        for smi, sc in beam[:6]
    ]
    print(f'Top beam — {op}:')
    display(MolsToGridImage(
        top_mols, molsPerRow=3, subImgSize=(300, 240),
        legends=top_labels
    ))

# Winners grouped by op
if winners_prot:
    print(f'\n{len(winners_prot)} total winner(s) — grouped by operation:')
    for op in OPS_PROT:
        op_wins = [(smi, sc) for smi, sc, o in winners_prot if o == op]
        if not op_wins:
            print(f'  {op}: no winners')
            continue
        print(f'\n── {op} winners ({len(op_wins)}) ──')
        for i, (smi, sc) in enumerate(op_wins[:9]):
            m = Chem.MolFromSmiles(smi)
            ts  = ecfp4_tanimoto(m)
            raw_shape = get_alignment_scores([m])[0]
            pa  = get_protected_atoms(m, PROTECT_SMARTS)
            print(f'  {i+1}. comb={sc:.4f}  shape={raw_shape:.4f}  TS={ts:.3f}  '
                  f'pa={len(pa)} atoms  {smi}')
        win_mols   = [Chem.MolFromSmiles(smi) for smi, _ in op_wins[:9]]
        win_labels = [
            f'comb={sc:.3f} TS={ecfp4_tanimoto(Chem.MolFromSmiles(smi)):.2f}'
            for smi, sc in op_wins[:9]
        ]
        display(MolsToGridImage(
            win_mols, molsPerRow=3, subImgSize=(300, 240),
            legends=win_labels,
        ))
else:
    print('No winners above threshold in protected run.')
    print('Best per lineage:')
    for op, beam in beams_prot.items():
        smi, sc = beam[0]
        print(f'  {op}: combined={sc:.4f}  {smi}')


### 10b. 3D overlay — protected optimisation best hits


In [ ]:
# Best winner per op (or best beam member), colour-coded by lineage.
# Reference: white sticks.  Analogues: per-op colour (same palette as Section 9).
overlay_entries_prot = []
for op in OPS_PROT:
    op_wins = [(smi, sc) for smi, sc, o in winners_prot if o == op]
    smi = op_wins[0][0] if op_wins else beams_prot[op][0][0]
    overlay_entries_prot.append((smi, op))

print('Overlaying reference (white) + best hit per lineage:')
for smi, op in overlay_entries_prot:
    m   = Chem.MolFromSmiles(smi)
    ts  = ecfp4_tanimoto(m)
    pa  = get_protected_atoms(m, PROTECT_SMARTS)
    print(f'  [{op}] TS={ts:.3f}  pa={len(pa)} atoms  colour={OP_COLOURS[op]}  {smi}')

view2 = py3Dmol.view(width=700, height=500)
view2.addModel(MolToMolBlock(ref_3d), 'mol')
view2.setStyle({'model': 0}, {'stick': {'color': 'white', 'radius': 0.12}})

for i, (smi, op) in enumerate(overlay_entries_prot):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    molblock, sc = best_aligned_molblock(mol)
    if molblock is None:
        continue
    colour = OP_COLOURS[op]
    view2.addModel(molblock, 'mol')
    view2.setStyle({'model': i + 1}, {'stick': {'color': colour, 'radius': 0.12}})

view2.zoomTo()
view2.show()


---
**Tips for improving results:**

- Increase `n_replacements` (25 → 50+) for broader exploration.
- Increase `generations` for deeper search from promising parents.
- Set `score_threshold=0.3` to pre-filter chemically implausible products.
- Use `protect_smarts='N1CCNCC1'` in the `OPS` lambdas to lock the piperazine core — see Section 10 for a full example.
